# Step 2: Port check (our attention patch vs the repo, then Qwen2.5-VL in English)

**Kaggle settings:** GPU **T4 x2** · Internet **on** · Inputs: `awsaf49/coco-2017-dataset`, your `xling-spatial-code` dataset,
and (optional) the output of notebook 01 (for the identical 80% test split).

Part A runs **LLaVA-1.5 through our patch** with the repo's exact VSR prompt and image resize. The accuracy should land close to
notebook 01 (within ~1–2 points; we score the Yes/No logit margin instead of searching the generated text, and run newer transformers).
Part B runs **Qwen2.5-VL** in English: does scaling image attention move accuracy the same way?

In [ ]:
# Shared setup for notebooks 02-04 (pinned transformers; do NOT run in the same session as notebook 01)
!pip install -q "transformers==4.57.1" "accelerate>=1.0" "huggingface_hub>=0.34"
import glob, os, sys
# The xling_spatial code: either a Kaggle dataset you uploaded (Add Input) or a git clone of your team repo.
REPO_URL = ""   # e.g. "https://github.com/<user>/<repo>.git" (leave empty if you attached the code as a Kaggle dataset)
def _find():
    return glob.glob("/kaggle/input/**/xling_spatial/__init__.py", recursive=True) + glob.glob("/kaggle/working/**/xling_spatial/__init__.py", recursive=True)
hits = _find()
if not hits and REPO_URL:
    !git clone -q {REPO_URL} /kaggle/working/repo
    hits = _find()
assert hits, "Set REPO_URL to your GitHub repo, or attach the 'xling-spatial-code' dataset (see README step 0)"
CODE_DIR = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, CODE_DIR); print("code:", CODE_DIR)

import torch, transformers, numpy as np, pandas as pd, matplotlib.pyplot as plt
from xling_spatial import data, runner, analysis, attention
from xling_spatial.models import VLM
from xling_spatial.tokenizer_stats import tokenizer_table
print(torch.__version__, transformers.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

COCO_ROOT = data.find_coco_root(); print("COCO:", COCO_ROOT)
WORK = "/kaggle/working"
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")   # needed only for gated models (Gemma 3)
except Exception:
    pass

def free(vlm):
    attention.uninstall()
    del vlm.model
    import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
vsr_test = data.load_vsr("test", COCO_ROOT)
vsr_val = data.load_vsr("val", COCO_ROOT)
print(len(vsr_val), len(vsr_test), vsr_test.subj.isna().mean(), "share without subj/obj")

# Reuse the repo's 80% test split if notebook 01's output is attached
idx = glob.glob("/kaggle/input/**/sampled_idx_VSR.npy", recursive=True)
if idx:
    val_idx = set(np.load(idx[0]).tolist())
    vsr_repo_test = vsr_test[[i not in val_idx for i in range(len(vsr_test))]].reset_index(drop=True)
    print("using repo split:", len(vsr_repo_test))
else:
    vsr_repo_test = vsr_test; print("notebook-01 output not attached; using all of VSR test")
boxes = data.load_coco_boxes(COCO_ROOT, pd.concat([vsr_val, vsr_test]).image.unique().tolist(), f"{WORK}/boxes_vsr.json")

## Part A: LLaVA-1.5-7B through our patch (should match notebook 01)

In [ ]:
llava = VLM("llava-hf/llava-1.5-7b-hf", dtype="float16")
print(llava.n_layers, "LM layers patched")
res_a = runner.run(llava, vsr_repo_test, ["en"], COCO_ROOT, f"{WORK}/step2a_llava15_vsr",
                   weights=(1.0, 0.5, 1.2), log_weights=(1.0,), raw_template="adaptvis_vsr",
                   preresize=(2300, 4096), boxes=None)

In [ ]:
base = res_a[np.isclose(res_a.weight, 1.0)]
av = runner.adaptvis_predictions(res_a, 0.5, 1.2, 0.64)
comp = pd.DataFrame([
    {"method": "base", "acc": base.correct.mean()},
    {"method": "scaling_vis w=0.5", "acc": res_a[np.isclose(res_a.weight, 0.5)].correct.mean()},
    {"method": "adapt_vis (0.5, 1.2, 0.64)", "acc": av.correct.mean(), "share_w1": (av.weight_used == 0.5).mean()},
])
comp.to_csv(f"{WORK}/step2a_ours_vs_repo.csv", index=False)
step1 = glob.glob("/kaggle/input/**/step1_adaptvis_repo_summary.csv", recursive=True)
display(comp); display(pd.read_csv(step1[0]) if step1 else "attach notebook 01 output to compare")
print("share of answers that are not yes/no (top token):", (~base.top_token.str.strip().str.lower().isin(["yes", "no"])).mean())

In [ ]:
free(llava)

## Part B: Qwen2.5-VL in English (VSR val → tune, VSR test → evaluate)

Start with **3B** (fits one T4 in fp16). Switch `QWEN` to the 7B model once everything runs; `device_map="auto"` splits it over both T4s.
If margins come out `NaN`, rerun with `dtype="float32"` (fp16 overflow).

In [ ]:
QWEN = "Qwen/Qwen2.5-VL-3B-Instruct"     # later: "Qwen/Qwen2.5-VL-7B-Instruct"
N_VAL, N_TEST = 300, 600                  # raise once timings look fine
WEIGHTS = (0.5, 0.8, 1.0, 1.2, 1.5, 2.0)
qwen = VLM(QWEN)
tag = QWEN.split("/")[-1]
res_val = runner.run(qwen, vsr_val.head(N_VAL), ["en"], COCO_ROOT, f"{WORK}/step2b_{tag}_val", weights=WEIGHTS, boxes=boxes)
res_test = runner.run(qwen, vsr_test.head(N_TEST), ["en"], COCO_ROOT, f"{WORK}/step2b_{tag}_test", weights=WEIGHTS,
                      boxes=boxes, save_maps=True)
print("NaN margins:", res_test.margin.isna().sum())

In [ ]:
best, grid = runner.tune_adaptvis(res_val)
print("AdaptVis params tuned on val:", best)
pd.Series(best).to_json(f"{WORK}/adaptvis_params_{tag}_en.json")
av_test = runner.adaptvis_predictions(res_test, best["weight1"], best["weight2"], best["threshold"])
best_w = analysis.summarize(res_val).sort_values("acc").iloc[-1].weight
table = pd.DataFrame([
    {"method": "base", "acc": res_test[np.isclose(res_test.weight, 1.0)].correct.mean()},
    {"method": f"scaling_vis w={best_w} (val-tuned)", "acc": res_test[np.isclose(res_test.weight, best_w)].correct.mean()},
    {"method": "adapt_vis (val-tuned)", "acc": av_test.correct.mean()},
])
table.to_csv(f"{WORK}/step2b_{tag}_summary.csv", index=False); table

In [ ]:
analysis.accuracy_vs_weight(res_test); plt.show()
analysis.confidence_by_lang(res_test); plt.show()
att = pd.read_csv(f"{WORK}/step2b_{tag}_test/attention.csv", dtype={"item_id": str})
analysis.plot_layers(att, title=f"{tag} · English VSR · last-token attention"); plt.savefig(f"{WORK}/step2b_{tag}_layers.png", dpi=150); plt.show()

**Done when:**
- Part A accuracy is close to notebook 01 for base, scaling_vis and adapt_vis (this shows the patch reproduces AdaptVis).
- Part B: val-tuned AdaptVis ≥ base on the test items, and `box_frac` (share of image attention on the two objects) sits above the dashed
  chance line in middle layers. If scaling does nothing or hurts for Qwen, that is a finding too: report it, since AdaptVis was only
  shown on LLaVA.